<a href="https://colab.research.google.com/github/GT0096/LLM_from_scratch/blob/main/SLM_Tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install ipdb tqdm wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 25.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [ipdb]


In [2]:
%%bash
nvidia-smi

Fri Oct 30 06:03:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   33C    P0             51W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
import os, sys, io
import ipdb # for debugging, variation of pdb
from tqdm import tqdm
from datetime import datetime
import platform, shutil #detect platform type
import requests, zipfile

os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
#pytorch
import torch
import torch.nn as nn
from torch.nn import functional as F

#tokenizer
import sentencepiece as spm

#these improves performance for Ampere architecture
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

#Empty GPU Cache Memory
torch.cuda.empty_cache()

In [4]:
# Download necessary files and create necessary folders
# wiki.txt -n dataset: a tiny segment of the English Wikipedia
# wiki_tokenizer.model: Trained tokenizer file
# wiki_tokenizer.vocab : Trained tokenizer file
# encoded_data.pt : dataset tokenized with tokenizer

files_url = "https://ideami.com/llm_train"
print("Downloading files using python")
response = requests.get(files_url)
zipfile.ZipFile(io.BytesIO(response.content)).extractall(".")

In [5]:
#Architecture Parameters
batch_size = 8 # Reduced from 64 to save memory
context = 512
embed_size = 384 #Each word is transformed by the tokenizer typpically into one OR MORE tokens,
                  #\\each of which has a numerical ID
n_layer = 7
n_heads = 7
BIAS = True

# Hyperparameter
lr = 3e-4
dropout = 0.05
weight_decay = 0.01
grad_clip = 1.0

#Training Parameters

train_iters = 100000
eval_interval = 50
eval_iter = 10
compile = False
load_pretrained = False
checkpoint_dir = 'models/'
checkpoint_fn = 'latest.pt'
checkpoint_load_fn = 'latest.pt'
dtype = torch.float16

# mode
inference = False

#Device
device = 'cuda' if torch.cuda.is_available() else "cpu"

print("Device: you wil be using: ", device)

Device: you wil be using:  cuda


In [6]:

# Logging

wandb_log = True
wandb_project = "llm_test"
wandb_run_name = "llm1-" + datetime.now().strftime("%Y_%m_%d_%H_%M_%S")

if wandb_log:
  import wandb
  wandb.init(project=wandb_project, name=wandb_run_name)

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: gaurav-prigaus (gaurav-prigaus-xoriant) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [7]:
with open('wiki.txt', 'r', encoding='utf-8') as f:
    text = f.read()
print(text[30000:30300])

terms.
For example, there are objects in two groups (as shown on the right). The objects are various shapes, where one group has 3 of them while the other has 2. When the two groups combine into one, the overall amount (sum) of the shapes become 5.

Vertical Addition

The animation above demonstrate


In [8]:
#Tokenizer

sp = spm.SentencePieceProcessor(model_file='wiki_tokenizer.model')

vocab_size = sp.get_piece_size()
print(f"Tokenizer vocab size: {vocab_size}")

Tokenizer vocab size: 4096


In [9]:
encode = lambda s : sp.Encode(s)
decode = lambda l : sp.Decode(l)

print(encode("Hello World"))
print(decode([312, 471, 4037, 870]))

[312, 471, 4037, 870]
Hello World


In [10]:
if os.path.exists(f"encoded_data.pt"):
  data = torch.load('encoded_data.pt')
else:
  data = torch.tensor(encode(text), dtype=torch.long)
  torch.save(data, 'encoded_data.pt')

In [11]:
data

tensor([4031,   13, 4061,  ...,   13,   13,   13])

In [12]:
data_size = len(data)
spl = int(0.9*data_size)
train_data = data[:spl]
val_data = data[spl:]

print(f"Total data size: {data_size}")
print(f"Train data size: {len(train_data)}")
print(f"Val data size: {len(val_data)}")

Total data size: 59211077
Train data size: 53289969
Val data size: 5921108


In [13]:
def get_batch(split):
  data = train_data if split=="train" else val_data
  inds = torch.randint(len(data)-context, (batch_size,))
  x = torch.stack([data[i: i+context] for i in inds]) #batch size  + sequence length
  y = torch.stack([data[i + 1 : i+ context + 1] for i in inds])

  x,y = x.to(device) , y.to(device)
  return x,y

x , y = get_batch('train')
print(x.shape, y.shape)
print(x[0][:10])
print(y[0][:10])

torch.Size([8, 512]) torch.Size([8, 512])
tensor([ 772,  259, 2783, 1861, 4051,  310, 2243,  302,  822,  289],
       device='cuda:0')
tensor([ 259, 2783, 1861, 4051,  310, 2243,  302,  822,  289, 2531],
       device='cuda:0')


# LLM Model



*   19 million parameters with the default configuration
*   Can be trained with 1 single GPU
*   With 64 batch size on 16 gb gpu




In [14]:
#################################################################################
################## LLM MODEL #############################################
# 19 million parameters with the default configuration
# Can be trained with 1 single GPU
# With 8 Batch Size, should require 4 GB of GPU Memory
# With 128 Batch Size, should require 24 GB of GPU Memory
# Adjust Batch Size as needed for less or more memory and training speed
# Because of small dataset and model, results will be limited but enough to
# demonstrate good improvement during the training and understand all the
# main technology involved in building LLMs
#################################################################################
###############################################
##################################

class GPT(nn.Module):

    def __init__(self):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size,embed_size) # Create embedding layer
        self.positions = nn.Embedding(context, embed_size) # Create basic positioning embeddings
        self.blocks = nn.Sequential(*[Block(n_heads) for _ in range(n_layer)]) # setup transformer blocks
        self.ln = nn.LayerNorm(embed_size) # normalization layers
        self.final_linear = nn.Linear(embed_size, vocab_size, bias=BIAS) # feedforward linear layer
        self.apply(self._init_weights) # Initialize the weights

    # Weights initialization
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            # Initialize weight matrices with normal distribution with mean 0 and small std
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            # Initialize bias parameters to 0
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        # Initialize Embedding weights with normal distribution with mean 0 and small std
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    # Running the LLM model
    def forward(self, input, targets=None):
        # BS = Batch Size / SL = Sequence Length or context length
        # For easier reading, I assume embedding dim of 384 and vocab size of 4096 in comments
        loss= None
        BS,SL = input.shape  # (BS,SL)
        emb = self.embeddings(input)  # (BS,SL,384)
        pos = self.positions(torch.arange(SL, device=device)) # (SL,384)
        x = emb+pos  # combine embedding and positioning stages (BS,SL,384)
        x = self.blocks(x)  #(BS,SL,384)
        x = self.ln(x) # (BS,SL,384)
        logits = self.final_linear(x) # (BS,SL,4096)

        # Calculate Loss if training with targets

        # Cross Entropy Logic
        # (equivalent to negative log likelihood)

        # Information: -log p(x) (inverse of probability)
        # Entropy: avg of information in random variable (prob distribution): - sum_x (x * log(x))
        # CrossEntropy: Compares 2 distr q(true) & p(predicted) in terms of information distance: -sum_x (q(x) * log p(x))
        # LLMs CrossEntropy: true labels are 1 for true, 0 for the rest, so it simplifies to: -sum_x log p(x)

        if targets is not None:
            BS, SL, VS = logits.shape  # (BS,SL,4096)
            logits = logits.view(BS*SL,VS)  # Reshape to prepare for cross_entropy (BS*SL,4096)
            targets = targets.view(BS*SL)   # Reshape as well (BS*SL)
            loss = F.cross_entropy(logits,targets)

            # Optional: Just for fun, manual way to calculate cross_entropy
            # By default, we comment out the manual version to prevent calculating the loss twice (will make things slower)

            # First apply softmax to produce probabilities
            #counts = logits.exp()  # (BS*SL,4096)
            #prob = counts / counts.sum(-1, keepdim=True) # (BS*SL,4096),(BS*SL,1) = (BS*SL,4096)
            #loss2 = -prob[torch.arange(BS*SL),targets].log().mean() # torch.arange(B*T) (BS*SL) | targets (BS*SL)

            # Finally at each of prob's positions, we pick the index specified by the respective target
            # example: targets[3]=329, prob[3][329] = 0.014

            # Most times they will match, sometimes they will not because F.cross_entropy is more precise
            # By uncommenting the following lines, you can see when they don't match
            #if ( not torch.allclose(loss,loss2)):
            #    print(f"[Loss Diff] Pytorch:{loss.item()} Manual:{loss2.item()}")

        return logits,loss

    # Generate a new sample
    def generate(self, input, max=500):
        # SL = Sequence Length or context length
        for _ in range(max): # until you reach the maximum number of tokens
            input = input[:,-context:] #(1, input length until max of SL)
            logits, _ = self(input)  # (1, input length, 4096)
            logits = logits[:,-1,:]  # Pick last probability discarding the dimension (1, 4096)
            probs = F.softmax(logits, dim=-1) # (1,4096)
            next = torch.multinomial(probs, num_samples=1) # Sample next token value
            input = torch.cat((input,next),dim=1) # Add new token to the input
        return input

In [15]:
########################################
##########Transformer Block Class ######
########################################

class Block(nn.Module):
    # A transformer block combines communication and computation over the data
    # Helps create complex processing and also emphasize relationships between the
    # members of the sequence through the attention mechanisms
    def __init__(self, n_heads):
        super().__init__()
        head_size = embed_size // n_heads # We split the embedding dimensions among the number of heads
        self.ma = Multihead(n_heads,head_size) # We setup the multihead system within each block
        self.feed_forward = ForwardLayer(embed_size)
        self.ln1 = nn.LayerNorm(embed_size) # Normalizing layer
        self.ln2 = nn.LayerNorm(embed_size) # Normalizing layer

        # LayerNorm normalizes the inputs across the features for each data point independently.
        # It subtracts the mean and divides by the standard deviation, followed by scaling and shifting.
        # It is computationally more intensive than for example RMSnorm but offers greater flexibility.

    def forward(self, x):
        x = x + self.ma(self.ln1(x))  # We normalize and then apply multi head attention
        x = x + self.feed_forward(self.ln2(x)) # we normalize again and then apply a feed forward layer
        return x


In [16]:
# The ForwardLayer applies a network that increases the computational complexity of the processing
class ForwardLayer(nn.Module):
    def __init__(self,embed_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(embed_size, 6*embed_size, bias=BIAS),
            nn.GELU(),
            nn.Linear(6*embed_size, embed_size, bias=BIAS),
            nn.Dropout(dropout)
        )
    def forward(self,x):
        x = self.network(x)
        return x

In [17]:
# Multihead Attention Layer
# This layer coordinates the different attention heads within each transformer block
class Multihead(nn.Module):
    def __init__(self,n_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(n_heads)]) # Setup the heads | head_size = embed_size // n_heads
        self.combine = nn.Linear(head_size * n_heads, embed_size, bias=BIAS) # (378,384) - in case of our default values
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # BS = Batch Size / SL = Sequence Length or context length
        # x is (BS,SL,384)  # 384 is default embed size
        x = torch.cat([head(x) for head in self.heads], dim=-1)
        # Each head outputs (BS,SL, head_size)
        # Combining them with torch.cat produces (BS,SL,378)  378 is default head_size * default n_heads = 54 * 7
        x = self.combine(x) # project them back to embed_size (BS, SL, 384)  384 is default embed_size
        x = self.dropout(x)
        return x

In [18]:
# Head Attention Layer
# Detects and reinforces patterns in relationships between members of sequence
class Head(nn.Module):
    # BS = Batch Size / SL = Sequence Length or context length
    def __init__(self, head_size):
        super().__init__()
        self.queries= nn.Linear(embed_size, head_size, bias=BIAS) # Query Projection (embed_dim, head_size) (384, 54)
        self.keys= nn.Linear(embed_size, head_size, bias=BIAS) # Key Projection (384, 54)
        self.values= nn.Linear(embed_size, head_size, bias=BIAS) # Value Projection (384, 54)
        # We declare a triangular matrix that we will use to mask future tokens from the current position
        # self.tril contains 0s in upper triangle and 1s in lower triangle + diagonal
        self.register_buffer('tril',torch.tril(torch.ones(context,context))) # self.tril - (SL,SL)
        self.dropout = nn.Dropout(dropout)

    def forward(self,x):
        BS,SL, VS = x.shape
        q=self.queries(x) # (BS,SL,54)  54 is the head_size
        k=self.keys(x) # (BS,SL,54)
        v=self.values(x) # (BS,SL,54)

        # Calculate square attention weights matrix with dot product of q and k, and normalize
        attn_w = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5 # (BS, SL, SL)

        # mask out future tokens, pay attention only to the past
        attn_w = attn_w.masked_fill(self.tril[:SL,:SL]==0, float('-inf'))  # set to -inf the upper right triangle of 0s

        attn_w = F.softmax(attn_w, dim=-1) # Transform into probabilities (BS, SL, SL)
        attn_w = self.dropout(attn_w) # (BS, SL, SL)

        # use attention weights to update the features of our tokens
        x = attn_w @ v # (BS,SL,54) # 54 is the head_size = embed_dim // n_heads
        return x

In [19]:
#################################################################################
# Main Training Process
#################################################################################

# Main Setup

model = GPT() # Instantiate LLM
model = model.to(dtype) # Set the precision type
model = model.to(device) # Move it to the right device

# Torch.compile compiles a PyTorch model to an optimized version, aiming to improve runtime performance and efficiency.
# Disable if your system doesn't support it
if compile:
    print("Torch :: Compiling model")
    model = torch.compile(model)


# Print the number of parameters of our model (19 million in our case)
print(sum(p.numel() for p in model.parameters()) / 1e6, " Million parameters")

19.837954  Million parameters


In [20]:
@torch.no_grad()
def generate_sample(input):
  t1 = torch.tensor(encode(input), dtype=torch.long, device=device)
  t1 = t1[None, :]
  newgen = model.generate(t1, max=64)[0].tolist()
  result = decode(newgen)
  print(f"{result}")

generate_sample("Once upon s time")

Once upon s timeir takingII bre O Mar�stant Cup* Washingtonaff Avrown ide Frank Army Way schoolzawood if Saintouri ground sisterid Province Sm writer themselves you�areut� August Eng track Virginia poiam cast Redander Us Kingdom party nine becomes magazineiev fun givesatives happen inhabitvelopillersennam Indiasurarg


In [21]:
# Training Setup

model = GPT()
model = model.to(dtype)
model =model.to(device)

if compile:
  print("Torch: Compile mode")
  model = torch.compile(model)

print(sum(p.numel() for p in model.parameters())/1e6, 'M parameters')


19.837954 M parameters


In [22]:
# Calculate the Loss
@torch.no_grad()  # Prevent gradient calculation
def calculate_loss():
    out={}
    model.eval()
    for split in ['train','eval']:
        l=torch.zeros(eval_iter)  # Create a tensor of zeros the size of eval_iters
        for i in range(eval_iter):
            x,y=get_batch(split) # Get a new batch of data
            _,loss=model(x,y)  # Calculate the loss
            l[i]=loss  # Store the loss in the next position of tensor
        out[split]=l.mean().item()  # Calculate the mean and extract the final value
    model.train()
    return out

l=calculate_loss()
print(l)

{'train': 8.353906631469727, 'eval': 8.356249809265137}


In [23]:
#setting up optimizer

p_dict = {p_name: p for p_name, p in model.named_parameters() if p.requires_grad}
weight_decay_p  = [p for n, p in p_dict.items() if p.dim() >=2]
no_weight_decay_p = [p for n, p in p_dict.items() if p.dim() < 2]

optimizer_groups = [
    {'params':weight_decay_p, 'weight_decay':weight_decay},
    {'params':no_weight_decay_p, 'weight_decay':0.0}
]
optimizer = torch.optim.AdamW(optimizer_groups, lr=lr, betas=(0.9,0.99))

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, train_iters, eta_min=lr/10)

start_iteration = 0
best_val_loss = float('inf') # track the best validation los value


In [24]:
# Loading Checkpoints
# Loads a previously saved checkpoint
def load_checkpoint(path):
    print("LLM - Loading model")
    checkpoint = torch.load(path)
    model.load_state_dict(checkpoint['model_state_dict']) # Load parameters
    optimizer.load_state_dict(checkpoint['optimizer_state_dict']) # Load optimizer state
    iteration = checkpoint['iteration'] # In what iteration did we save the model?
    loss = checkpoint['loss'] # What was the last loss value?
    print(f"Loaded iter {iteration} with loss {loss}")
    return iteration, loss

################# OPTIONAL : LOAD A PREVIOUS CHECKPOINT
if os.path.exists(f"{checkpoint_dir}/{checkpoint_load_fn}") and load_pretrained:
    start_iteration, loss = load_checkpoint(checkpoint_dir + checkpoint_load_fn)
    best_val_loss = loss

In [25]:
from pickle import TRUE
# Inference

# inference = True
if inference == True:
  model.eval()
  while True:
    qs = input("Enter text (q to quit): ")
    if qs == "":
      continue
    if qs == "q":
      break
    generate_sample(qs)

In [ ]:
#################################################################
###################### TRAINING #################################
#################################################################

# Fix: Cast model back to float32 to prevent float16 NaN overflows
model = model.to(torch.float32)

try:
    for i in tqdm(range(start_iteration, train_iters)):
        xb,yb = get_batch("train") # Get a new batch of data
        logits,loss = model(xb,yb) # Run the LLM and get the logits and the loss

        if (i % eval_interval==0 or i == train_iters-1): # Calculate the loss
            l = calculate_loss()
            print(f"\n{i}: train loss: {l['train']} / val loss: {l['eval']}")

            # We do a quick test so that we observe the evolution through the training
            # Remember that we use a very small dataset which doesn't include all topics
            generate_sample("The mountain in my city is") # Generate a sample

            if l['eval'] < best_val_loss: # If we improved the best loss, save a checkpoint
                best_val_loss = l['eval']
                print("[CHECKPOINT]: Saving with loss: ", best_val_loss)
                torch.save({
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'loss': best_val_loss,
                    'iteration': i,
                }, checkpoint_dir + checkpoint_fn)

            if wandb_log:
                wandb.log({
                        "loss/train": l['train'],
                        "loss/val": l['eval'],
                        "lr": scheduler.get_last_lr()[0],
                    },
                    step = i)

        optimizer.zero_grad(set_to_none=True) # Reset gradients
        loss.backward() # Calculate new gradients

        # This line clips the gradients to prevent the exploding gradient problem during training.
        # Exploding gradients can occur when gradients become too large, causing unstable updates to model weights.
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)

        optimizer.step() # Update the model parameters
        scheduler.step() # Update the learning rate value

    if wandb_log:
        wandb.finish()


except KeyboardInterrupt:
    print("Training interrupted. Cleaning up...")

finally:
    # Release GPU memory
    torch.cuda.empty_cache()
    print("GPU memory released.")

if wandb_log:
    wandb.finish()
torch.cuda.empty_cache()

# Code designed by Javier ideami
# ideami.com

  0%|          | 0/100000 [00:00<?, ?it/s]


0: train loss: 8.35200309753418 / val loss: 8.363802909851074
The mountain in my city isiro Jerital<iles� sent L born readinois candid inh saysresent September coll song activ import."A en Sovietita eventuallyet separateitch lived sec groundals British living� Film later Lee teamsves thousject human Recordschool speak actingoth They called oppos cond were performanceusesZ techn songuel University civil Elizabeth Columb
[CHECKPOINT]: Saving with loss:  8.363802909851074


  0%|          | 50/100000 [00:10<3:56:51,  7.03it/s]


50: train loss: 5.994660377502441 / val loss: 6.056696891784668
The mountain in my city is Len of M nowandterically day mostly living plant stat into 1a performanceag, "N. Firstayson of many campaignWhenured to 1, andoldajor val come.
Aub.chestraran, partostupt Mad a city toJording River types scored degree orig
[CHECKPOINT]: Saving with loss:  6.056696891784668


  0%|          | 100/100000 [00:20<3:50:58,  7.21it/s]


100: train loss: 5.643857002258301 / val loss: 5.655456066131592
The mountain in my city is a steyatheris. "
W participy independentgest example in the ealped, " MahouayederalCics any than 20avagniut might585xs can there 28nds, 145. The st been teamunose Duts to
[CHECKPOINT]: Saving with loss:  5.655456066131592


... (751 outputs left out)


10850: train loss: 2.7091217041015625 / val loss: 2.8429059982299805


 11%|█         | 10851/100000 [36:39<27:22:12,  1.11s/it]

The mountain in my city is a law in Nigeria. The story on this to learn about the style. He is a lot of famous random culture. He has a long Titan called "Niger". The writings are about blacks and metal.

In Huditionel

The term for Kodron


 11%|█         | 10888/100000 [36:45<4:04:39,  6.07it/s]